# Normal-estimation research progress

This notebook reads the canonical experiment records from `experiments/normal-estimation/records/` and visualizes development and validation RMSE. Its main chart follows the style of Karpathy's autoresearch progress plot: every measured attempt, highlighted retained candidates, and a stepwise running best. Lower RMSE is better.

Re-run all cells after the research loop completes another iteration. The notebook does not evaluate estimators or modify research state.

In [ ]:
from __future__ import annotations

import json
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%config InlineBackend.figure_format = 'retina'

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "experiments"
        ).is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the point-cloud-central repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXPERIMENT_DIR = REPO_ROOT / "experiments" / "normal-estimation"
RECORD_DIR = EXPERIMENT_DIR / "records"
CONDITIONS = ("clean", "low", "medium", "high", "stripe", "gradient")
STATUS_COLORS = {
    "keep": "#2ecc71",
    "provisional": "#f39c12",
    "discard": "#95a5a6",
    "crash": "#e74c3c",
}

record_paths = sorted(RECORD_DIR.glob("[0-9][0-9][0-9][0-9].json"))
records = [json.loads(path.read_text()) for path in record_paths]
if not records:
    raise FileNotFoundError(f"No canonical records found in {RECORD_DIR}")

latest = records[-1]
validated = latest["validated"]
frontier = latest["frontier"]
print(f"Loaded {len(records)} experiments through #{latest['iteration']}")
print(
    f"Validated RMSE: dev={validated['dev_rmse']:.6f}, val={validated['val_rmse']:.6f}"
)
print(f"Frontier commit: {frontier['commit'][:7]} ({frontier['status']})")

## Overall RMSE progress

Gray points are discarded candidates, orange points are provisional candidates, green points are validated keeps, and red crosses mark crashes at the top edge because they have no score. The green step line is the best validated RMSE known after each experiment. Hollow diamonds show validation measurements on the development plot.

In [ ]:
def numeric(record: dict[str, object], key: str) -> float:
    value = record.get(key)
    return np.nan if value is None else float(value)


iterations = np.array([int(record["iteration"]) for record in records])
development = np.array([numeric(record, "dev_rmse") for record in records])
validation = np.array([numeric(record, "val_rmse") for record in records])
statuses = np.array([str(record["status"]) for record in records])

validated_best = []
best = np.nan
for record in records:
    if record["status"] == "keep" and record.get("dev_rmse") is not None:
        candidate = float(record["dev_rmse"])
        best = candidate if np.isnan(best) else min(best, candidate)
    validated_best.append(best)

fig, ax = plt.subplots(figsize=(16, 8), layout="constrained")
measured = np.isfinite(development)

for status, label, marker, size in (
    ("discard", "Discarded", "o", 52),
    ("provisional", "Provisional", "o", 72),
    ("keep", "Validated keep", "o", 82),
):
    selected = measured & (statuses == status)
    if selected.any():
        ax.scatter(
            iterations[selected],
            development[selected],
            c=STATUS_COLORS[status],
            s=size,
            marker=marker,
            edgecolors="black" if status == "keep" else "white",
            linewidths=0.7,
            alpha=0.95,
            label=label,
            zorder=4,
        )

validated_points = np.isfinite(validation)
ax.scatter(
    iterations[validated_points],
    development[validated_points],
    facecolors="none",
    edgecolors="#2980b9",
    marker="D",
    s=150,
    linewidths=1.5,
    label="Validation measured",
    zorder=5,
)

ax.step(
    iterations,
    validated_best,
    where="post",
    color="#1e8449",
    linewidth=2.4,
    alpha=0.85,
    label="Running validated best",
    zorder=3,
)

finite_values = development[measured]
span = max(float(np.ptp(finite_values)), 0.5)
top = float(np.max(finite_values) + 0.12 * span)
crashed = statuses == "crash"
if crashed.any():
    ax.scatter(
        iterations[crashed],
        np.full(crashed.sum(), top),
        c=STATUS_COLORS["crash"],
        marker="x",
        s=75,
        linewidths=2,
        label="Crash (no RMSE)",
        zorder=4,
    )

for record in records:
    if record["status"] != "keep" or record.get("dev_rmse") is None:
        continue
    description = textwrap.shorten(
        str(record["description"]), width=44, placeholder="…"
    )
    ax.annotate(
        description,
        (int(record["iteration"]), float(record["dev_rmse"])),
        xytext=(7, 8),
        textcoords="offset points",
        fontsize=8.5,
        color="#196f3d",
        rotation=24,
        ha="left",
        va="bottom",
    )

ax.set(
    xlabel="Experiment #",
    ylabel="Development RMSE in degrees (lower is better)",
    title=(
        f"Normal-estimation progress: {len(records)} experiments, "
        f"{sum(statuses == 'keep')} validated keeps"
    ),
)
ax.set_xticks(iterations)
ax.grid(True, alpha=0.2)
ax.legend(loc="upper right", ncols=2, fontsize=9)
plt.show()

## Validation RMSE

Validation is intentionally sparse. This chart only compares candidates that the trusted controller actually promoted to validation. The horizontal line marks the original fixed-k PCA baseline.

In [ ]:
validated_records = [record for record in records if record.get("val_rmse") is not None]
val_iterations = np.array([int(record["iteration"]) for record in validated_records])
val_scores = np.array([float(record["val_rmse"]) for record in validated_records])
val_statuses = [str(record["status"]) for record in validated_records]
running_validation_best = np.minimum.accumulate(val_scores)

fig, ax = plt.subplots(figsize=(16, 6), layout="constrained")
ax.scatter(
    val_iterations,
    val_scores,
    c=[STATUS_COLORS[status] for status in val_statuses],
    s=85,
    edgecolors="black",
    linewidths=0.6,
    zorder=4,
)
ax.step(
    val_iterations,
    running_validation_best,
    where="post",
    color="#1e8449",
    linewidth=2.4,
    label="Running validation best",
)
ax.axhline(
    float(records[0]["val_rmse"]),
    color="#34495e",
    linestyle="--",
    linewidth=1.3,
    label="Fixed-k PCA baseline",
)
for record in validated_records:
    ax.annotate(
        f"#{record['iteration']} {record['status']}",
        (int(record["iteration"]), float(record["val_rmse"])),
        xytext=(5, 6),
        textcoords="offset points",
        fontsize=8,
    )
ax.set(
    xlabel="Experiment #",
    ylabel="Validation RMSE in degrees (lower is better)",
    title="Promoted candidates on the validation tier",
)
ax.set_xticks(iterations)
ax.grid(True, alpha=0.2)
ax.legend()
plt.show()

## Per-condition development progress

These six panels expose accuracy trade-offs hidden by the equally weighted aggregate. Lines connect measured experiments only; crashes are omitted because they have no scores.

In [ ]:
condition_records = [
    record
    for record in records
    if isinstance(record.get("condition_rmse"), dict)
    and isinstance(record["condition_rmse"].get("development"), dict)
]
condition_iterations = np.array(
    [int(record["iteration"]) for record in condition_records]
)

fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex=True, layout="constrained")
for ax, condition in zip(axes.flat, CONDITIONS, strict=True):
    scores = np.array(
        [
            float(record["condition_rmse"]["development"][condition])
            for record in condition_records
        ]
    )
    ax.plot(condition_iterations, scores, color="#bdc3c7", linewidth=1.2, zorder=1)
    for status in ("discard", "provisional", "keep"):
        selected = np.array(
            [record["status"] == status for record in condition_records]
        )
        if selected.any():
            ax.scatter(
                condition_iterations[selected],
                scores[selected],
                c=STATUS_COLORS[status],
                s=42 if status == "discard" else 60,
                edgecolors="black" if status == "keep" else "white",
                linewidths=0.5,
                zorder=3,
            )
    ax.axhline(
        float(records[0]["condition_rmse"]["development"][condition]),
        color="#34495e",
        linestyle="--",
        linewidth=1,
        alpha=0.7,
    )
    ax.set_title(condition.capitalize())
    ax.set_ylabel("RMSE (degrees)")
    ax.grid(True, alpha=0.2)

for ax in axes[-1]:
    ax.set_xlabel("Experiment #")
    ax.set_xticks(iterations)
fig.suptitle("Development RMSE by PCPNet condition", fontsize=15)
plt.show()

## Accuracy-runtime frontier

A point is Pareto-efficient when no measured candidate is both faster and more accurate. This makes the cost of retained mechanisms visible alongside RMSE.

In [ ]:
measured_records = [
    record
    for record in records
    if record.get("dev_rmse") is not None and record.get("runtime_s") is not None
]
runtimes = np.array([float(record["runtime_s"]) for record in measured_records])
scores = np.array([float(record["dev_rmse"]) for record in measured_records])
pareto = np.array(
    [
        not np.any(
            (runtimes <= runtime)
            & (scores <= score)
            & ((runtimes < runtime) | (scores < score))
        )
        for runtime, score in zip(runtimes, scores, strict=True)
    ]
)

fig, ax = plt.subplots(figsize=(11, 7), layout="constrained")
ax.scatter(
    runtimes,
    scores,
    c=[STATUS_COLORS[str(record["status"])] for record in measured_records],
    s=65,
    edgecolors="white",
    linewidths=0.7,
    alpha=0.95,
)
pareto_order = np.argsort(runtimes[pareto])
ax.plot(
    runtimes[pareto][pareto_order],
    scores[pareto][pareto_order],
    color="#8e44ad",
    linewidth=2,
    marker="o",
    label="Pareto frontier",
)
for record, runtime, score in zip(measured_records, runtimes, scores, strict=True):
    ax.annotate(
        f"#{record['iteration']}",
        (runtime, score),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )
ax.set(
    xlabel="Development runtime (seconds)",
    ylabel="Development RMSE in degrees (lower is better)",
    title="Accuracy-runtime trade-off",
)
ax.grid(True, alpha=0.2)
ax.legend()
plt.show()

## Static progress image

Run the cell below to write a repository-root `progress.png` similar to autoresearch. It redraws the overall chart without changing research state. The image is ignored by Git.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8), layout="constrained")
for status, label, size in (
    ("discard", "Discarded", 52),
    ("provisional", "Provisional", 72),
    ("keep", "Validated keep", 82),
):
    selected = measured & (statuses == status)
    if selected.any():
        ax.scatter(
            iterations[selected],
            development[selected],
            c=STATUS_COLORS[status],
            s=size,
            edgecolors="black" if status == "keep" else "white",
            linewidths=0.7,
            label=label,
            zorder=4,
        )
ax.step(
    iterations,
    validated_best,
    where="post",
    color="#1e8449",
    linewidth=2.4,
    label="Running validated best",
)
if crashed.any():
    ax.scatter(
        iterations[crashed],
        np.full(crashed.sum(), top),
        c=STATUS_COLORS["crash"],
        marker="x",
        s=75,
        linewidths=2,
        label="Crash (no RMSE)",
    )
for record in records:
    if record["status"] != "keep" or record.get("dev_rmse") is None:
        continue
    ax.annotate(
        textwrap.shorten(str(record["description"]), width=44, placeholder="…"),
        (int(record["iteration"]), float(record["dev_rmse"])),
        xytext=(7, 8),
        textcoords="offset points",
        fontsize=8.5,
        color="#196f3d",
        rotation=24,
    )
ax.set(
    xlabel="Experiment #",
    ylabel="Development RMSE in degrees (lower is better)",
    title=f"Normal-estimation progress: {len(records)} experiments",
)
ax.set_xticks(iterations)
ax.grid(True, alpha=0.2)
ax.legend(loc="upper right", ncols=2, fontsize=9)
output_path = REPO_ROOT / "progress.png"
fig.savefig(output_path, dpi=180, bbox_inches="tight")
plt.close(fig)
output_path